# 🚨 Notebook 3: Compensating Transactions

The hardest part of sagas: writing the **undo** step for every real step.
Undo is *not* the same as rollback — it's a new, forward transaction that reverses business effect.

## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
class SagaError(Exception): pass

state = {'stock': 10, 'charged': 0, 'shipment': None, 'notified': False}

def reserve_stock(): state['stock'] -= 1
def release_stock(): state['stock'] += 1

def charge(): state['charged'] = 20
def refund(): state['charged'] = 0; print('    💸 refund issued')

def ship():
    raise SagaError('courier down')
def cancel_ship(): state['shipment'] = None

def run(steps):
    done = []
    for name, do, undo in steps:
        try:
            print(f'→ {name}'); do(); done.append((name, undo))
        except SagaError as e:
            print(f'✗ {name} failed: {e}')
            for n, u in reversed(done):
                print(f'  ↶ compensating {n}'); u()
            return 'compensated'
    return 'done'

result = run([
    ('reserve_stock', reserve_stock, release_stock),
    ('charge',        charge,        refund),
    ('ship',          ship,          cancel_ship),
])
print('result:', result, 'state:', state)


### Real-world gotchas
- **Idempotent** compensations: they may run twice.
- **Partial visibility**: customers may see the stock reserved briefly.
- Some actions cannot be undone (email sent, SMS sent). Prefer **semantic compensation** (send apology email).